# Trace Count v6: Separator Trace, No Numeric Indices

v6 copies the controlled v2 marker-trace experiment, but removes numeric trace indices from the thinking trace.

```text
v2 thinking trace: <Think/> <1> <A> <2> <B> <3> <C> </Think> <Ans> <3>
v6 thinking trace: <Think/> <Sep> <A> <Sep> <B> <Sep> <C> </Think> <Ans> <3>
```

The final-answer metric still checks only the numeric token after `<Ans>`. The core question is whether separator-trace thinking still learns item-specific retrieval/counting once explicit prefix-count tokens are gone.


## 1. Environment and Repo Setup


In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
REPO_DIR = Path("/content/Synthetic_CoT_NiaH_Count")
PULL_REPO = True
REPAIR_NUMPY_ABI = True
INSTALL_MINIMAL_DEPS = False
INSTALL_EDITABLE_PACKAGE = False

if Path("/content").exists():
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
        if PULL_REPO and (REPO_DIR / ".git").exists():
            subprocess.run(["git", "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
        os.chdir(REPO_DIR)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from scripts.colab_setup import setup_colab

ROOT = setup_colab(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    pull=False,
    repair_numpy_abi=REPAIR_NUMPY_ABI,
    install_deps=INSTALL_MINIMAL_DEPS,
    install_editable=INSTALL_EDITABLE_PACKAGE,
)

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("cwd:", Path.cwd())
print("python:", sys.executable)
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 2. Runtime Settings

Use `debug` first for an end-to-end sanity check. For the real v6 run, set `PRESET = "main"`.

`RUN_PROBES` and `RUN_ATTENTION` can be turned off for a faster first pass, then rerun later with `STAGE = "probes"` or `STAGE = "attention"` after checkpoints exist.


In [ ]:
PRESET = "debug"  # "debug" or "main"
STAGE = "all"      # "all", "train", "final_eval", "probes", "attention", "plots", "report"
CONFIG = f"synthetic_counting_v6/configs/{PRESET}.yaml"
RUN_DIR = f"runs/v6_separator_trace_{PRESET}_seed1234"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SKIP_COMPLETED = True
RUN_PROBES = True
RUN_ATTENTION = True

# Optional overrides. Leave as None to use YAML defaults.
TRAIN_STEPS = None
BATCH_SIZE = None
VAL_EXAMPLES_PER_COUNT = None
TEST_EXAMPLES_PER_COUNT = None
PROBE_EXAMPLES_PER_COUNT = None
ATTENTION_EXAMPLES_PER_COUNT = None

settings = {
    "PRESET": PRESET,
    "STAGE": STAGE,
    "CONFIG": CONFIG,
    "RUN_DIR": RUN_DIR,
    "DEVICE": DEVICE,
    "SKIP_COMPLETED": SKIP_COMPLETED,
    "RUN_PROBES": RUN_PROBES,
    "RUN_ATTENTION": RUN_ATTENTION,
}
settings


## 3. Run v6 Pipeline


In [ ]:
cmd = [
    sys.executable,
    "-u",
    "-m",
    "synthetic_counting_v6.run_v6_experiment",
    "--config",
    CONFIG,
    "--stage",
    STAGE,
    "--run-dir",
    RUN_DIR,
    "--device",
    DEVICE,
]
if SKIP_COMPLETED:
    cmd.append("--skip-completed")
if not RUN_PROBES:
    cmd.append("--no-probes")
if not RUN_ATTENTION:
    cmd.append("--no-attention")
for flag, value in [
    ("--train-steps", TRAIN_STEPS),
    ("--batch-size", BATCH_SIZE),
    ("--val-examples-per-count", VAL_EXAMPLES_PER_COUNT),
    ("--test-examples-per-count", TEST_EXAMPLES_PER_COUNT),
    ("--probe-examples-per-count", PROBE_EXAMPLES_PER_COUNT),
    ("--attention-examples-per-count", ATTENTION_EXAMPLES_PER_COUNT),
]:
    if value is not None:
        cmd += [flag, str(value)]

print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
captured = []
for line in proc.stdout:
    print(line, end="")
    captured.append(line.rstrip())
returncode = proc.wait()
if returncode:
    print("---- Last 160 log lines ----")
    print("\n".join(captured[-160:]))
    raise subprocess.CalledProcessError(returncode, cmd)

FINAL_RUN_DIR = Path(RUN_DIR)
FINAL_REPORT = FINAL_RUN_DIR / "report" / "report.html"
for line in reversed(captured):
    if line.startswith("FINAL_RUN_DIR "):
        FINAL_RUN_DIR = Path(line.split(" ", 1)[1].strip())
    if line.startswith("FINAL_REPORT "):
        FINAL_REPORT = Path(line.split(" ", 1)[1].strip())
print("FINAL_RUN_DIR =", FINAL_RUN_DIR)
print("FINAL_REPORT =", FINAL_REPORT)


## 4. Result Tables

Accuracy here always means exact match of the final numeric count token after `<Ans>`. For `thinking_sep_trace / generated_trace`, the model must first generate the `<Sep> marker ... </Think> <Ans>` suffix by itself; for `oracle_trace_final_readout`, the gold separator trace is provided and only final answer readout is tested.


In [ ]:
RUN_DIR_PATH = Path(FINAL_RUN_DIR) if "FINAL_RUN_DIR" in globals() else Path(RUN_DIR)
metrics_dir = RUN_DIR_PATH / "metrics"

for filename in ["metrics_train.csv", "metrics_eval_by_bin.csv", "metrics_eval_by_count.csv", "metrics_final_test_by_bin.csv", "metrics_final_test_by_count.csv"]:
    path = metrics_dir / filename
    if path.exists():
        display(Markdown(f"**{filename}**"))
        df = pd.read_csv(path)
        display(df.tail(20))
    else:
        display(Markdown(f"Missing `{path}`"))


## 5. Figures

Each figure caption names the x-axis, y-axis, and group definition. The most important behavioral comparison is final-count accuracy for `non_thinking / direct`, `thinking_sep_trace / generated_trace`, and `thinking_sep_trace / oracle_trace_final_readout`.


In [ ]:
figure_specs = [
    ("plots/train_loss_vs_step.png", "x = training step; y = masked completion loss. Non-thinking supervises only final answer/EOS; separator-thinking supervises trace + final answer, so raw completion loss lengths differ."),
    ("plots/eval_final_answer_loss_vs_step.png", "x = training step; y = final-answer cross-entropy restricted to numeric answer tokens. This is the most comparable loss curve."),
    ("plots/eval_accuracy_by_bin_vs_step.png", "x = training step; y = exact final-count accuracy. Groups are model/eval mode/count bin: low=1-3, mid=4-6, high=7-10."),
    ("plots/final_accuracy_by_count.png", "x = gold count 1-10; y = exact final-count accuracy at final checkpoint. Groups compare direct non-thinking, generated separator trace, and oracle trace readout."),
    ("plots/accuracy_heatmap_by_count_and_step_non_thinking.png", "Non-thinking heatmap: x = training step, y = gold count, color = exact final-count accuracy."),
    ("plots/accuracy_heatmap_by_count_and_step_thinking_generated_trace.png", "Generated separator trace heatmap: x = training step, y = gold count, color = exact final-count accuracy after free generation."),
    ("plots/accuracy_heatmap_by_count_and_step_thinking_oracle_trace.png", "Oracle separator trace heatmap: x = training step, y = gold count, color = final readout accuracy when gold trace is supplied."),
    ("plots/trace_exact_by_count.png", "x = gold count; y = rate that generated trace exactly equals `<Sep> marker_1 ... <Sep> marker_n`."),
    ("plots/trace_delimiter_count_accuracy_by_count.png", "x = gold count; y = rate that number of generated `<Sep>` delimiters equals the true count."),
    ("attention/attention_thinking_sep_correct_top1_by_layer_head.png", "Separator-thinking attention: rows = layer, columns = head, color = whether sep_token_k attends most to kth prompt needle."),
    ("attention/attention_matrix_thinking_sep_best_head_mid.png", "Best separator retrieval head: x = prompt needle index j, y = trace item index k, color = average attention mass."),
    ("probes/probe_prefix_count_accuracy_heatmap_thinking_sep_trace.png", "Probe heatmap: x = layer, y = trace anchor, color = linear decodability of prefix count k."),
    ("probes/probe_sep_token_prefix_probe_minus_position_baseline.png", "x = layer; y = sep_token_k prefix probe accuracy minus position-only baseline. Positive value suggests information beyond absolute position."),
]
for rel, caption in figure_specs:
    path = RUN_DIR_PATH / rel
    if path.exists():
        display(Markdown(f"**{rel}**  \n{caption}"))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Missing `{rel}`"))


## 6. Interpretation Notes

- If `thinking_sep_trace / generated_trace` reaches high final accuracy and high trace exactness, repeated `<Sep>` delimiters are enough; v2's numeric trace indices were not necessary for the main behavior.
- If oracle trace final readout is high but generated trace accuracy is low, the model can read count from a correct trace but struggles to generate the de-indexed trace.
- If `sep_token_k` has diagonal attention to the kth prompt needle, that is the v6 analogue of v2's targeted retrieval head.
- If prefix-count probe accuracy disappears after subtracting the position-only baseline, do not interpret it as a clean counter direction; trace position can still leak k even without numeric trace tokens.


## 7. Save Results to Google Drive


In [ ]:
SAVE_TO_DRIVE = True
DRIVE_DEST_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results")

if SAVE_TO_DRIVE and Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dest = DRIVE_DEST_ROOT / f"v6_separator_trace_{PRESET}_seed1234_{timestamp}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUN_DIR_PATH, dest, dirs_exist_ok=True)
    display(Markdown(f"Saved results to Google Drive: `{dest}`"))
else:
    display(Markdown(f"SAVE_TO_DRIVE skipped. Local run dir: `{RUN_DIR_PATH}`"))


## 8. Optional Auto-Disconnect Runtime


In [ ]:
AUTO_DISCONNECT_AFTER_SAVE = False

if AUTO_DISCONNECT_AFTER_SAVE and Path("/content").exists():
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print("Could not auto-disconnect runtime:", exc)
else:
    print("Auto-disconnect skipped.")
